# Atlantic Salmon Growth Model Walkthrough

This notebook is the presentation layer for task 5. The reusable implementation stays in `stata/salmon_growth.py`, while this notebook makes it easier to explain the model, inspect the calibration, and hover over the growth curves during an interview.

## Model

The port preserves the executed Stata logic:

- $SGR(\%/day) = \alpha \cdot W^{\beta} \cdot e^{-k(T - T_{opt})^2}$
- $W_{t+1} = W_t \cdot e^{SGR(T, W_t)/100}$

The original comments say the 14C calibration target is 1.53 % per day at 150 g, but because the code uses $T_{opt}=13.5$, the effective value at 14C is slightly lower. The notebook keeps that nuance visible instead of hiding it.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

repo_root = Path.cwd()
if not (repo_root / 'stata').exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from stata.salmon_growth import GrowthModelConfig, TEMPERATURE_COLORS, run_and_export

config = GrowthModelConfig()
output_dir = repo_root / 'stata' / 'output'
result = run_and_export(
    config,
    output_dir,
    write_svg_chart=False,
    write_html_chart=True,
)
long_df = pd.read_csv(result['artifacts']['long_csv'])
summary = result['summary']
summary_df = (
    pd.DataFrame(summary['curve_summary'])
    .sort_values('final_weight_g', ascending=False)
    .reset_index(drop=True)
)
summary_df

,temperature_c,temperature_label,final_weight_g,mean_sgr_percent,max_daily_gain_g,day_to_500g,day_to_1000g
0,14.0,14C,3397.458,1.0547,20.2700,157.0,229.0
1,12.0,12C,3118.145,1.0332,18.4138,163.0,238.0
2,16.0,16C,2633.584,0.9910,15.2244,175.0,255.0
3,10.0,10C,2058.375,0.9294,11.5011,195.0,285.0
4,18.0,18C,1501.935,0.8506,7.9875,225.0,329.0
5,8.0,8C,1035.945,0.7578,5.1446,269.0,393.0
6,20.0,20C,686.592,0.6549,3.1084,334.0,NaN
7,6.0,6C,446.147,0.5472,1.7868,NaN,NaN
8,4.0,4C,194.537,0.3396,0.5421,NaN,NaN


In [2]:
validation_df = pd.DataFrame(summary['validation'])
best_curve = summary['best_temperature_by_final_weight']

print(
    f"Best final weight: {best_curve['temperature_label']} -> {best_curve['final_weight_g']:.3f} g"
)
validation_df

Best final weight: 14C -> 3397.458 g


,name,passed,expected,observed,delta
0,alpha_matches_stata_formula,True,1.53,1.53,NaN
1,effective_14c_sgr_is_close_to_comment_target,True,1.53,1.52313,-0.00687
2,all_weights_positive,True,NaN,NaN,NaN
3,14c_outperforms_10c_and_18c,True,NaN,"{'10C': 2058.375, '14C': 3397.458, '18C': 1501...",NaN


In [ ]:
from IPython.display import HTML, display



figure = go.Figure()



for temperature_label, curve_df in long_df.groupby('temperature_label', sort=False):

    temperature_c = float(curve_df['temperature_c'].iloc[0])

    line_width = 4.5 if temperature_c == config.calibration_temperature_c else 2.5

    figure.add_trace(

        go.Scatter(

            x=curve_df['day'],

            y=curve_df['weight_g'],

            mode='lines',

            name=temperature_label,

            line={'color': TEMPERATURE_COLORS.get(temperature_c, '#102A43'), 'width': line_width},

            customdata=curve_df[['sgr_percent', 'daily_gain_g']].to_numpy(),

            hovertemplate=(

                'Temperature=%{fullData.name}<br>'

                'Day=%{x}<br>'

                'Weight=%{y:.2f} g<br>'

                'SGR=%{customdata[0]:.3f}%/day<br>'

                'Daily gain=%{customdata[1]:.2f} g<extra></extra>'

            ),

        )

    )



figure.update_layout(

    title='Atlantic salmon growth by temperature',

    template='plotly_white',

    hovermode='x unified',

    xaxis_title='Day post sea-transfer',

    yaxis_title='Weight (g)',

    legend_title='Temperature',

    title_x=0.02,

    margin={'l': 60, 'r': 30, 't': 80, 'b': 60},

)

figure.add_annotation(

    x=0.02,

    y=1.08,

    xref='paper',

    yref='paper',

    showarrow=False,

    align='left',

    text='Hover over each curve to inspect day-level weight, SGR, and daily gain. The 14C line is emphasized because it is the calibration scenario.',

)



display(HTML(figure.to_html(include_plotlyjs='cdn', full_html=False)))


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [4]:
artifact_rows = [{'artifact': name, 'path': str(path)} for name, path in result['artifacts'].items()]
pd.DataFrame(artifact_rows)

,artifact,path
0,wide_csv,f:\barents-lice-forecasting\stata\output\weigh...
1,long_csv,f:\barents-lice-forecasting\stata\output\weigh...
2,summary_json,f:\barents-lice-forecasting\stata\output\growt...
3,html_chart,f:\barents-lice-forecasting\stata\output\growt...
